In [ ]:
!pip install streamlit graphviz pyngrok
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.6.1) ...
Setting up cloudflared (2026.6.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
%%writefile app.py
import streamlit as st
from graphviz import Digraph

symbol_table = set()

# ---------------- LOGIC ---------------- #

def tokenize(sentence):
    return sentence.lower().strip().split()

def update_symbol_table(tokens):
    keywords = ["add","subtract","multiply","if","then","print","greater","less","and","from"]
    for token in tokens:
        if token.isalpha() and token not in keywords:
            symbol_table.add(token)

def generate_TAC(tokens):
    try:
        if tokens[0] == "add":
            return f"t1 = {tokens[1]} + {tokens[3]}"
        elif tokens[0] == "subtract":
            return f"t1 = {tokens[3]} - {tokens[1]}"
        elif tokens[0] == "multiply":
            return f"t1 = {tokens[1]} * {tokens[3]}"
        else:
            return "Invalid Syntax"
    except:
        return "Error"

def evaluate_expression(tokens):
    try:
        if tokens[0] == "add":
            return int(tokens[1]) + int(tokens[3])
        elif tokens[0] == "subtract":
            return int(tokens[3]) - int(tokens[1])
        elif tokens[0] == "multiply":
            return int(tokens[1]) * int(tokens[3])
    except:
        return "Execution Error"

def generate_parse_tree(tokens):
    dot = Digraph()
    dot.attr(bgcolor="#f8fafc")

    if tokens[0] == "add":
        dot.node("ADD", "ADD", style="filled", fillcolor="#bfdbfe")
        dot.node("A", tokens[1])
        dot.node("B", tokens[3])
        dot.edge("ADD","A")
        dot.edge("ADD","B")

    elif tokens[0] == "subtract":
        dot.node("SUB", "SUB", style="filled", fillcolor="#bbf7d0")
        dot.node("A", tokens[1])
        dot.node("B", tokens[3])
        dot.edge("SUB","B")
        dot.edge("SUB","A")

    elif tokens[0] == "multiply":
        dot.node("MUL", "MUL", style="filled", fillcolor="#bae6fd")
        dot.node("A", tokens[1])
        dot.node("B", tokens[3])
        dot.edge("MUL","A")
        dot.edge("MUL","B")

    return dot

# ---------------- UI ---------------- #

st.set_page_config(layout="wide")

st.markdown("""
<h1 style='text-align:center;
font-size:40px;
background: linear-gradient(to right, #6366f1, #ec4899);
-webkit-background-clip: text;
color: transparent;'>
🧠 Natural Language Compiler
</h1>
""", unsafe_allow_html=True)

st.markdown("### 💬 Enter Commands")

sentence = st.text_area(
    "Commands",
    """add 5 and 3
multiply 4 and 2
subtract 3 from 10""",
    height=150
)

run = st.button("🚀 Run Compiler")

# ---------------- EXECUTION ---------------- #

if run:

    lines = sentence.strip().split("\n")
    results = []

    for line in lines:
        tokens = tokenize(line)

        if not tokens:
            continue

        update_symbol_table(tokens)
        tac = generate_TAC(tokens)
        result = evaluate_expression(tokens)

        results.append({
            "input": line,
            "tokens": tokens,
            "tac": tac,
            "result": result,
            "tree": generate_parse_tree(tokens)
        })

    for res in results:

        st.markdown("---")

        col1, col2 = st.columns(2)

        with col1:
            st.markdown("### 📥 Input")
            st.info(res["input"])

            st.markdown("### ⚙️ TAC")
            st.code(res["tac"])

            st.markdown("### 🧮 Result")
            st.success(res["result"])

        with col2:
            st.markdown("### 🔍 Tokens")
            st.write(res["tokens"])

            st.markdown("### 📊 Symbol Table")
            st.write(list(symbol_table))

        st.markdown("### 🌳 Parse Tree")
        st.graphviz_chart(res["tree"])

Overwriting app.py


In [ ]:
import subprocess
import time
import re

# Kill previous runs
!pkill -f streamlit
!pkill -f cloudflared

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501"])

print("⏳ Starting Streamlit...")
time.sleep(8)

# Start tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("⏳ Generating public link...\n")

for line in tunnel.stdout:
    print(line)

    if "trycloudflare.com" in line:
        url = re.search(r"(https://[^\s]+)", line)
        if url:
            print("\n🚀 FINAL LINK:", url.group(1))
            break

⏳ Starting Streamlit...
⏳ Generating public link...

2026-06-24T07:01:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps

2026-06-24T07:01:41Z INF Requesting new quick Tunnel on trycloudflare.com...

2026-06-24T07:01:45Z INF +--------------------------------------------------------------------------------------------+

2026-06-24T07:01:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |

2026-06-24T07:01:45Z INF